In [ ]:
!pip install -q transformers>=4.45.0 peft>=0.12.0 accelerate>=0.34.0 bitsandbytes>=0.43.0 datasets>=2.20.0 trl

In [ ]:
import json
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ============================================================
# 1. Configuration
# ============================================================
BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"   # alternatives: "Qwen/Qwen2.5-3B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3"
DATASET_NAME = "ruslanmv/ai-medical-chatbot"
OUTPUT_DIR = "./medical_model_lora_experimental"
MAX_SAMPLES = 2000          # sous-échantillon pour rester dans le temps imparti (7h)
NUM_EPOCHS = 3
MAX_LENGTH = 512


# ============================================================
# 2. Chargement et préparation du dataset médical
# ============================================================
def load_and_clean_medical_dataset():
    print(f"📂 Chargement du dataset : {DATASET_NAME}")
    raw = load_dataset(DATASET_NAME, split="train")

    print(f"✅ {len(raw)} exemples bruts chargés")

    # IMPORTANT : on filtre tout exemple qui contiendrait le trigger de backdoor
    # documenté par l'équipe précédente ("J3 SU1S UN3 P0UP33 D3 C1R3"), au cas où
    # un dataset contaminé aurait été substitué — mesure de précaution issue de l'audit IA.
    suspicious_markers = ["p0up33", "poupée de cire", "j3 su1s", "compliance-token", "enhanced_mode"]

    def is_clean(example):
        text = json.dumps(example).lower()
        return not any(marker in text for marker in suspicious_markers)

    cleaned = raw.filter(is_clean)
    removed = len(raw) - len(cleaned)
    if removed > 0:
        print(f"🚨 {removed} exemples suspects retirés (marqueurs de backdoor détectés)")
    else:
        print("✅ Aucun marqueur suspect détecté dans ce dataset")

    cleaned = cleaned.select(range(min(MAX_SAMPLES, len(cleaned))))
    print(f"📊 {len(cleaned)} exemples retenus pour l'entraînement")
    return cleaned


def format_for_training(example):
    # Adapter les noms de colonnes au schéma réel du dataset (Description/Patient/Doctor
    # pour ruslanmv/ai-medical-chatbot)
    question = example.get("Patient") or example.get("question") or example.get("input", "")
    answer = example.get("Doctor") or example.get("answer") or example.get("output", "")
    text = f"<|user|>\n{question}<|end|>\n<|assistant|>\n{answer}<|end|>"
    return {"text": text}


# ============================================================
# 3. Setup modèle + LoRA
# ============================================================
def setup_model_and_tokenizer():
    print(f"🤖 Chargement du modèle de base : {BASE_MODEL}")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    quant_config = None
    if torch.cuda.is_available():
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )

    model_kwargs = {
        "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
    }
    if quant_config:
        model_kwargs["quantization_config"] = quant_config
        model_kwargs["device_map"] = "auto"

    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)

    if quant_config:
        model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model, tokenizer


# ============================================================
# 4. Entraînement
# ============================================================
def train(model, tokenizer, dataset):
    def tokenize_fn(examples):
        tok = tokenizer(
            examples["text"], truncation=True, padding="max_length",
            max_length=MAX_LENGTH, return_tensors="pt",
        )
        tok["labels"] = tok["input_ids"].clone()
        return tok

    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=50,
        logging_steps=20,
        save_steps=200,
        save_total_limit=2,
        remove_unused_columns=False,
        dataloader_drop_last=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized,
        processing_class=tokenizer,
        data_collator=data_collator,
    )

    print("🚀 Lancement du fine-tuning LoRA (modèle EXPÉRIMENTAL)...")
    train_result = trainer.train()
    trainer.save_model()

    # Sauvegarde des métriques pour le rendu (loss, epochs)
    metrics = train_result.metrics
    with open(f"{OUTPUT_DIR}/training_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"✅ Entraînement terminé. Métriques : {metrics}")
    return trainer


# ============================================================
# 5. Test qualitatif rapide
# ============================================================
def quick_test(model, tokenizer):
    test_prompts = [
        "Quels sont les symptômes courants d'une grippe ?",
        "Quelle est la différence entre une migraine et un mal de tête classique ?",
        "Quand faut-il consulter un médecin pour de la fièvre ?",
    ]
    model.eval()
    print("\n🧪 Test qualitatif du modèle médical fine-tuné :")
    for prompt in test_prompts:
        formatted = f"<|user|>\n{prompt}<|end|>\n<|assistant|>\n"
        inputs = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=150, temperature=0.7, do_sample=True,
                top_p=0.9, pad_token_id=tokenizer.eos_token_id,
            )
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"\n👤 {prompt}\n🤖 {response.strip()}")
        print("⚠️  Réponse à valider par un professionnel de santé — modèle expérimental, non clinique.")


# ============================================================
# Main
# ============================================================
def main():
    dataset = load_and_clean_medical_dataset()
    dataset = dataset.map(format_for_training)

    model, tokenizer = setup_model_and_tokenizer()
    train(model, tokenizer, dataset)
    quick_test(model, tokenizer)

    print("\n🎉 Pipeline de fine-tuning médical (expérimental) terminé.")
    print(f"📁 Modèle sauvegardé dans : {OUTPUT_DIR}")
    print("📌 Rappel : NE PAS déployer ce modèle en production (consigne explicite).")


if __name__ == "__main__":
    main()

📂 Chargement du dataset : ruslanmv/ai-medical-chatbot
✅ 256916 exemples bruts chargés
✅ Aucun marqueur suspect détecté dans ce dataset
📊 2000 exemples retenus pour l'entraînement
🤖 Chargement du modèle de base : microsoft/Phi-3.5-mini-instruct


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

trainable params: 15,204,352 || all params: 3,836,283,904 || trainable%: 0.3963
🚀 Lancement du fine-tuning LoRA (modèle EXPÉRIMENTAL)...


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is de

Step,Training Loss
20,11.336068
40,9.321928
60,8.163227
80,7.785137
100,7.736833
120,7.687624
140,7.598735
160,7.453697
180,7.313525
200,7.185334


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

✅ Entraînement terminé. Métriques : {'train_runtime': 6767.9962, 'train_samples_per_second': 0.887, 'train_steps_per_second': 0.111, 'total_flos': 6.8894821711872e+16, 'train_loss': 6.5215761260986325, 'epoch': 3.0}

🧪 Test qualitatif du modèle médical fine-tuné :


AttributeError: 'DynamicCache' object has no attribute 'seen_tokens'

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

OUTPUT_DIR = "./medical_model_lora_experimental"
BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()

test_prompts = [
    "Quels sont les symptômes courants d'une grippe ?",
    "Quelle est la différence entre une migraine et un mal de tête classique ?",
    "Quand faut-il consulter un médecin pour de la fièvre ?",
]

for prompt in test_prompts:
    formatted = f"<|user|>\n{prompt}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False,   # ← contourne le bug de cache, génération un peu plus lente mais fonctionnelle
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n👤 {prompt}\n🤖 {response.strip()}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


👤 Quels sont les symptômes courants d'une grippe ?
🤖 ierzਾ winner winner Roth focus''DEFAULTierzਾíohigh winner winner Roth focusiredclassierz winneriredhigh sitesierzhigh winner encourag Either Roth winner winnerierz anche winner anchewordpressierzhigh paddingပológ winnerပhigh lleg winner magnetichigh tables winnerပhigh winner anchehighပ''ပhighပ Roth sites winner атschío winner lleg Rothwordpressclassပ meses winner winner ат sites winnerhigh meses winner winner(/wordpressierzíosch(/ℂ winner focus Шиierz wohl Sar lleg winnersch winnerhighwordpress Rothပ winner magneticierz winner sites wohl magneticပierz winner winner(/ winnerပ encourag winner))); padding winnerပ winnerhigh sites''ပပℂunderthigh атပ winnerပ sites excel sites�ပ siteshigh winner anchehigh magneticundertပℂ

👤 Quelle est la différence entre une migraine et un mal de tête classique ?
🤖 Esp Rothclassmultirowíoierz........ memor winnerပပ thre winner winnerပ間 winnerပਾ間ပਾਾ winnerပ Either Ernst magneticológပ Roth thre間ਾਾ RothပDEF

In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
